Notebook examines parameter recapitulation using synthetic data.

In [3]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import scMPRAforge as scm

2025-07-28 13:34:41.858421: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-28 13:34:41.864600: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-07-28 13:34:41.864627: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [4]:
#autoreload for dev
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
#load ground truth
ground_truth_mu=pd.read_csv("../../notebooks/demos/parameter_extraction_demo/synthetic_ground_truth_carp.tsv",sep="\t")
ground_truth_mu=ground_truth_mu.set_index(['cell_type','cre_id'])
ground_truth_mu.head()

true_mu
cell_type cre_id             
brain     reference         1
          mediumbody       10
          everybody       114
          redgene          30
          neurogene        99

In [6]:
#start a lightweighrt cluster
from dask.distributed import Client, LocalCluster
cluster=LocalCluster()
client = Client(cluster)

2025-07-28 13:35:52,511 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:33477' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'_label_tensorzinb_regressors-0c85d47568056534fbcbedf8d7689e9c', '_smart_matrix-ed99a1d8f1860ac2c601139c5ae76fad', '_extract_zi-c9f1417c775878d8ef87f8a1fc45bff0', '_smart_matrix-b9f3238375b324ce40228bb9e8e8f66e', '_extract_theta-8c72e2f4ec3334f10e75d4269598c248', '_label_tensorzinb_regressors-20d8881b83323604e1aa8d8b3a874f58', '_extract_zi-27fcd1af429b94ec4a507de258c2db4e', '_extract_mu-ffb3e045ca6ed8f8c2eaa100812ec903', '_label_tensorzinb_regressors-0e75f7ae0faa05a9fcdb4b237ae61fe6', '_extract_mu-13567686f9835e1fe9c84fd340094abd', '_extract_zi-d92b6ee2427d9d00611bfbfe2490c2b8', '_extract_mu-f03268381f814f86b06af9c1b533b98d', '_extract_theta-7d8f81b08a7386c76a64ca5a15c53949', '_extract_mu-c2eaa96cfba9fb05878b44c099644d63', '_extract_mu-29be1977aa6c45951ff2bdaa2dcead03', '_extract_mu-bead037435095aa7

# Parameter extraction

In [7]:
dat=scm.scMPRA_data.from_tsv("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres_v3.tsv")
dat.ortho_filter()
dat.set_negative_controls(["nobody","weak"])
dat.set_reference_cell("liver")

primordial=scm.ortho()
primordial.criss_cross(client=client,dat=dat)
primordial.extract_params(client)

scMPRAforge: INFO: Dropped 0 of 27 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [8]:
scm.versus_truth(ground_truth_mu,primordial)

,by,mean_squared_error,mean_biased_error,mean_absolute_percent_error
0,cell_type,2.786826,-0.239707,11.786365
1,cre_id,2.633289,-0.268082,11.418997


In [9]:
cluster.close()